# Generación de Combinaciones y Precálculo

Este notebook toma el mejor modelo entrenado (desde MLflow) y genera estimaciones de precio para **todas las combinaciones únicas** de vehículos del dataset simulando distintos kilometrajes. Los resultados se guardan en una base **SQLite** rápida (`precalc.db`).

In [1]:
import os
import sqlite3
import pandas as pd
import numpy as np
import mlflow.pyfunc
from itertools import product

# Configurar MLflow local para que sepa dónde buscar
mlflow.set_tracking_uri("sqlite:///mlflow.db")

## 1. Cargar el Modelo desde MLflow
Descargaremos directamente la versión etiquetada como **Production**.

In [2]:
model_name = "ValorAuto_Model"
stage = "Production"
model_uri = f"models:/{model_name}/{stage}"

print(f"Cargando modelo desde MLflow Registry: {model_uri}...")
try:
    model = mlflow.pyfunc.load_model(model_uri)
    print("¡Modelo cargado exitosamente!")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")

Cargando modelo desde MLflow Registry: models:/ValorAuto_Model/Production...
¡Modelo cargado exitosamente!


## 2. Extraer Combinaciones Únicas
Cargaremos el dataset base para ver qué combinaciones de (Marca, Año, Tipo, etc.) existen realmente en el mercado.

In [3]:
input_path = '../data/vehicles/vehicles_clean.csv'
if not os.path.exists(input_path):
    print(f"No se encontró el archivo base en {input_path}")
else:
    print("Cargando dataset base...")
    df = pd.read_csv(input_path)
    
    # Extraer combinaciones únicas
    cat_features = ['year', 'manufacturer', 'fuel', 'transmission', 'drive', 'type']
    unique_combinations = df[cat_features].drop_duplicates().dropna()
    print(f"Encontramos {len(unique_combinations)} combinaciones únicas reales de vehículos.")

Cargando dataset base...
Encontramos 10128 combinaciones únicas reales de vehículos.


## 3. Simulación de Kilometraje
Multiplicaremos cada auto por 5 escenarios de kilometraje comunes.

In [4]:
odometer_ranges = [10000, 50000, 100000, 150000, 200000]
print(f"Generando simulaciones ({len(odometer_ranges)} escenarios por auto)...")

dfs = []
for odo in odometer_ranges:
    df_odo = unique_combinations.copy()
    df_odo['odometer'] = odo
    dfs.append(df_odo)
    
df_predict = pd.concat(dfs, ignore_index=True)

# Ordenar columnas como las espera el modelo
features = ['year', 'odometer', 'manufacturer', 'fuel', 'transmission', 'drive', 'type']
X_predict = df_predict[features]
print(f"Total de filas a predecir: {len(X_predict)}")

Generando simulaciones (5 escenarios por auto)...
Total de filas a predecir: 50640


## 4. Inferencia por Lotes (Batch Inference) y Guardado
Haremos las predicciones de una sola vez y volcaremos todo en SQLite para consumo en milisegundos.

In [5]:
print("Calculando precios masivamente...")
predictions = model.predict(X_predict)

df_result = X_predict.copy()
df_result['predicted_price'] = np.round(predictions, 2)

# Guardar en SQLite
os.makedirs('../data', exist_ok=True)
db_path = '../data/precalc.db'
print(f"Guardando {len(df_result)} registros en la base SQLite: {db_path}...")

conn = sqlite3.connect(db_path)
df_result.to_sql('precalculated_prices', conn, if_exists='replace', index=False)

# Crear índice para acelerar búsquedas
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_search ON precalculated_prices (manufacturer, year, type)")
conn.commit()
conn.close()

print("¡Base de datos generada y lista para el backend!")

Calculando precios masivamente...
Guardando 50640 registros en la base SQLite: ../data/precalc.db...
¡Base de datos generada y lista para el backend!
